# CRM Access Governance & Customer Data Protection
## Stage 9 — Analytical Data Model

This notebook converts the governance framework into a structured **analytical data model**.

The objective is to create a Power BI-ready star schema that integrates:

- CRM access events;
- access governance;
- contextual risk;
- data quality;
- privacy-safe customer analytics;
- governance rules;
- lineage-ready dimensions.

### Main Goals

1. Define the analytical grain.
2. Build a fact table for CRM access events.
3. Create reusable dimensions.
4. Build a privacy-safe customer dimension.
5. Create a governance rule dimension.
6. Add Data Quality and governance attributes to the fact.
7. Design star-schema relationships.
8. Create analytical metrics and validation checks.
9. Export Power BI-ready tables.
10. Prepare the project for Stage 10 — Governance Monitoring in Power BI.

> **Important:** this is a simulated analytical model built from synthetic data and project-defined governance rules.


In [1]:
# 1. Libraries and Settings

import pandas as pd
import numpy as np
import hashlib
from pathlib import Path

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 300)

DATA_PATH = Path("Permission_Aware_CRM_Governance_Synthetic_50000.csv")

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

print("Environment ready.")


Environment ready.


# 2. Load CRM Access Source

The original CRM access dataset is loaded as the transactional source.


In [2]:
access = pd.read_csv(DATA_PATH)

print(f"Rows: {access.shape[0]:,}")
print(f"Columns: {access.shape[1]}")
display(access.head())


Rows: 50,000
Columns: 15


,User_ID,Role,Region,Lead_Source,CRM_Action,Daily_Logins,Failed_Logins,Access_Hour,Device_Type,Data_Sensitivity,Policy_Compliance_Score,Anomaly_Score,Permission_Granted,Governance_Score,Access_Decision
0,1,Admin,North,Referral,ViewLead,6,1,10,Managed,5,76.19,0.114,True,51.81,Block
1,2,Sales Rep,West,Web,ViewLead,5,1,12,Managed,5,62.87,0.464,True,38.40,Block
2,3,Sales Rep,South,Partner,EditLead,6,1,20,Managed,1,81.87,0.083,False,48.57,Block
3,4,Analyst,South,Campaign,ViewLead,10,0,10,Managed,3,85.89,0.181,True,57.53,Block
4,5,Sales Rep,West,Web,EditLead,6,2,17,Managed,1,92.31,0.309,False,44.58,Block


# 3. Analytical Grain

The main fact table grain is:

> **One row per CRM access event**

Each row represents one observed CRM access attempt/event and contains:

- who the user is;
- what action was requested;
- device context;
- sensitivity;
- risk indicators;
- permission status;
- governance outcome;
- Data Quality monitoring fields.

This grain supports access-governance monitoring without aggregating away important risk signals.


# 4. Recreate Core Governance Logic

The Stage 3 governance logic is reconstructed so the analytical fact includes the same governance attributes.


In [3]:
df = access.copy()

df["Is_Blocked"] = df["Access_Decision"].eq("Block").astype(int)

df["High_Sensitivity_Flag"] = df["Data_Sensitivity"].ge(4)
df["BYOD_Flag"] = df["Device_Type"].eq("BYOD")
df["Failed_Login_Flag"] = df["Failed_Logins"].gt(0)

anomaly_threshold = df["Anomaly_Score"].quantile(0.90)
df["High_Anomaly_Flag"] = df["Anomaly_Score"].ge(anomaly_threshold)

min_hour = df["Access_Hour"].min()
max_hour = df["Access_Hour"].max()

df["Edge_Hour_Flag"] = (
    df["Access_Hour"].le(min_hour + 1) |
    df["Access_Hour"].ge(max_hour - 1)
)

df["Low_Compliance_Flag"] = df["Policy_Compliance_Score"].lt(
    df["Policy_Compliance_Score"].quantile(0.25)
)

df["Low_Governance_Flag"] = df["Governance_Score"].lt(
    df["Governance_Score"].quantile(0.25)
)

contextual_flags = [
    "High_Sensitivity_Flag",
    "BYOD_Flag",
    "Failed_Login_Flag",
    "High_Anomaly_Flag",
    "Edge_Hour_Flag",
    "Low_Compliance_Flag",
    "Low_Governance_Flag"
]

df["Contextual_Risk_Score"] = df[contextual_flags].sum(axis=1)

risk_level_map = {
    0: "LOW",
    1: "LOW",
    2: "MEDIUM",
    3: "MEDIUM",
    4: "HIGH",
    5: "HIGH",
    6: "CRITICAL",
    7: "CRITICAL"
}

df["Contextual_Risk_Level"] = df["Contextual_Risk_Score"].map(risk_level_map)

print(f"High anomaly threshold: {anomaly_threshold:.3f}")


High anomaly threshold: 0.389


# 5. Recreate Role × CRM Action Access Matrix

This dimension supports baseline authorization analysis.


In [4]:
roles = ["Admin", "Manager", "Sales Rep", "Support", "Analyst"]

actions = [
    "ViewLead",
    "EditLead",
    "CreateOpportunity",
    "ApproveDiscount",
    "ExportCRM",
    "DeleteLead"
]

access_matrix = pd.DataFrame(index=roles, columns=actions)

access_matrix.loc["Admin"] = [
    "ALLOW", "ALLOW", "ALLOW",
    "ALLOW", "ALLOW", "ALLOW"
]

access_matrix.loc["Manager"] = [
    "ALLOW", "ALLOW", "ALLOW",
    "ALLOW", "REVIEW", "REVIEW"
]

access_matrix.loc["Sales Rep"] = [
    "ALLOW", "ALLOW", "ALLOW",
    "REVIEW", "REVIEW", "BLOCK"
]

access_matrix.loc["Support"] = [
    "ALLOW", "REVIEW", "BLOCK",
    "BLOCK", "BLOCK", "BLOCK"
]

access_matrix.loc["Analyst"] = [
    "ALLOW", "BLOCK", "BLOCK",
    "BLOCK", "REVIEW", "BLOCK"
]

access_matrix_long = (
    access_matrix
    .reset_index()
    .rename(columns={"index": "Role"})
    .melt(
        id_vars="Role",
        var_name="CRM_Action",
        value_name="Baseline_Authorization"
    )
)

display(access_matrix_long.head(20))


,Role,CRM_Action,Baseline_Authorization
0,Admin,ViewLead,ALLOW
1,Manager,ViewLead,ALLOW
2,Sales Rep,ViewLead,ALLOW
3,Support,ViewLead,ALLOW
4,Analyst,ViewLead,ALLOW
5,Admin,EditLead,ALLOW
6,Manager,EditLead,ALLOW
7,Sales Rep,EditLead,ALLOW
8,Support,EditLead,REVIEW
9,Analyst,EditLead,BLOCK


# 6. Apply Baseline Authorization and Governance Decision


In [5]:
df = df.merge(
    access_matrix_long,
    on=["Role", "CRM_Action"],
    how="left"
)

def governance_decision(row):
    if row["Permission_Granted"] == False:
        return "BLOCK", "AUTH-001"

    if row["Baseline_Authorization"] == "BLOCK":
        return "BLOCK", "AUTH-002"

    if row["Contextual_Risk_Level"] == "CRITICAL":
        return "BLOCK", "CTX-001"

    if (
        row["High_Sensitivity_Flag"]
        and row["BYOD_Flag"]
        and row["High_Anomaly_Flag"]
    ):
        return "BLOCK", "CTX-002"

    if row["Baseline_Authorization"] == "REVIEW":
        return "REVIEW", "AUTH-003"

    if row["Contextual_Risk_Level"] == "HIGH":
        return "REVIEW", "CTX-003"

    if (
        row["High_Sensitivity_Flag"]
        and row["Failed_Login_Flag"]
    ):
        return "REVIEW", "CTX-004"

    if row["Contextual_Risk_Level"] == "MEDIUM":
        return "REVIEW", "CTX-005"

    return "ALLOW", "DEFAULT-001"

decision_output = df.apply(
    governance_decision,
    axis=1,
    result_type="expand"
)

decision_output.columns = [
    "Proposed_Access_Decision",
    "Triggered_Rule_ID"
]

df = pd.concat([df, decision_output], axis=1)

display(
    df[
        [
            "Role",
            "CRM_Action",
            "Permission_Granted",
            "Baseline_Authorization",
            "Contextual_Risk_Level",
            "Proposed_Access_Decision",
            "Triggered_Rule_ID"
        ]
    ].head(20)
)


,Role,CRM_Action,Permission_Granted,Baseline_Authorization,Contextual_Risk_Level,Proposed_Access_Decision,Triggered_Rule_ID
0,Admin,ViewLead,True,ALLOW,MEDIUM,REVIEW,CTX-004
1,Sales Rep,ViewLead,True,ALLOW,HIGH,REVIEW,CTX-003
2,Sales Rep,EditLead,False,ALLOW,LOW,BLOCK,AUTH-001
3,Analyst,ViewLead,True,ALLOW,LOW,ALLOW,DEFAULT-001
4,Sales Rep,EditLead,False,ALLOW,MEDIUM,BLOCK,AUTH-001
5,Analyst,ViewLead,True,ALLOW,MEDIUM,REVIEW,CTX-004
6,Support,ViewLead,True,ALLOW,LOW,ALLOW,DEFAULT-001
7,Analyst,CreateOpportunity,True,BLOCK,LOW,BLOCK,AUTH-002
8,Sales Rep,ExportCRM,False,REVIEW,MEDIUM,BLOCK,AUTH-001
9,Manager,EditLead,True,ALLOW,MEDIUM,REVIEW,CTX-005


# 7. Recreate Data Quality Monitoring Fields

The full Stage 4 rule engine does not need to be repeated here.

For the analytical model, we include the main record-level outputs needed for reporting:

- `Failed_Rule_Count`
- `DQ_Status`


In [6]:
dq_failures = pd.DataFrame(index=df.index)

dq_failures["User_ID_Missing"] = df["User_ID"].isna()
dq_failures["Role_Missing"] = df["Role"].isna()
dq_failures["Action_Missing"] = df["CRM_Action"].isna()
dq_failures["Permission_Missing"] = df["Permission_Granted"].isna()
dq_failures["Decision_Missing"] = df["Access_Decision"].isna()

dq_failures["Role_Invalid"] = ~df["Role"].isin(roles)
dq_failures["Action_Invalid"] = ~df["CRM_Action"].isin(actions)
dq_failures["Device_Invalid"] = ~df["Device_Type"].isin(["Managed", "BYOD"])
dq_failures["Hour_Invalid"] = ~df["Access_Hour"].between(0, 23)
dq_failures["Sensitivity_Invalid"] = ~df["Data_Sensitivity"].between(1, 5)
dq_failures["Anomaly_Invalid"] = ~df["Anomaly_Score"].between(0, 1)
dq_failures["Compliance_Invalid"] = ~df["Policy_Compliance_Score"].between(0, 100)
dq_failures["Governance_Invalid"] = ~df["Governance_Score"].between(0, 100)
dq_failures["Daily_Logins_Invalid"] = df["Daily_Logins"].lt(0)
dq_failures["Failed_Logins_Invalid"] = df["Failed_Logins"].lt(0)

dq_failures["Permission_Decision_Inconsistent"] = (
    (df["Permission_Granted"] == False) &
    (df["Access_Decision"] != "Block")
)

df["Failed_Rule_Count"] = dq_failures.sum(axis=1)

def dq_status(count):
    if count == 0:
        return "PASS"
    elif count <= 2:
        return "WARNING"
    return "FAIL"

df["DQ_Status"] = df["Failed_Rule_Count"].apply(dq_status)

display(
    df[
        ["User_ID", "Failed_Rule_Count", "DQ_Status"]
    ].head()
)


,User_ID,Failed_Rule_Count,DQ_Status
0,1,0,PASS
1,2,0,PASS
2,3,0,PASS
3,4,0,PASS
4,5,0,PASS


# 8. Create Surrogate Event Key

The fact table receives a stable analytical event key.


In [7]:
df = df.reset_index(drop=True)

df["Access_Event_Key"] = np.arange(
    1,
    len(df) + 1
)

display(
    df[
        ["Access_Event_Key", "User_ID", "CRM_Action"]
    ].head()
)


,Access_Event_Key,User_ID,CRM_Action
0,1,1,ViewLead
1,2,2,ViewLead
2,3,3,EditLead
3,4,4,ViewLead
4,5,5,EditLead


# 9. Dimension — Role


In [8]:
dim_role = (
    df[["Role"]]
    .drop_duplicates()
    .sort_values("Role")
    .reset_index(drop=True)
)

dim_role["Role_Key"] = np.arange(
    1,
    len(dim_role) + 1
)

dim_role = dim_role[
    ["Role_Key", "Role"]
]

display(dim_role)


,Role_Key,Role
0,1,Admin
1,2,Analyst
2,3,Manager
3,4,Sales Rep
4,5,Support


# 10. Dimension — CRM Action


In [9]:
action_risk_map = {
    "ViewLead": "Low",
    "EditLead": "Medium",
    "CreateOpportunity": "Medium",
    "ApproveDiscount": "High",
    "ExportCRM": "High",
    "DeleteLead": "Critical"
}

dim_action = (
    df[["CRM_Action"]]
    .drop_duplicates()
    .sort_values("CRM_Action")
    .reset_index(drop=True)
)

dim_action["Action_Key"] = np.arange(
    1,
    len(dim_action) + 1
)

dim_action["Action_Risk_Level"] = (
    dim_action["CRM_Action"]
    .map(action_risk_map)
)

dim_action["Is_Critical_Action"] = (
    dim_action["CRM_Action"]
    .isin(["ExportCRM", "DeleteLead", "ApproveDiscount"])
)

dim_action = dim_action[
    [
        "Action_Key",
        "CRM_Action",
        "Action_Risk_Level",
        "Is_Critical_Action"
    ]
]

display(dim_action)


,Action_Key,CRM_Action,Action_Risk_Level,Is_Critical_Action
0,1,ApproveDiscount,High,True
1,2,CreateOpportunity,Medium,False
2,3,DeleteLead,Critical,True
3,4,EditLead,Medium,False
4,5,ExportCRM,High,True
5,6,ViewLead,Low,False


# 11. Dimension — Device


In [10]:
device_control_map = {
    "Managed": "Organization Managed",
    "BYOD": "User Owned"
}

dim_device = (
    df[["Device_Type"]]
    .drop_duplicates()
    .sort_values("Device_Type")
    .reset_index(drop=True)
)

dim_device["Device_Key"] = np.arange(
    1,
    len(dim_device) + 1
)

dim_device["Device_Control_Type"] = (
    dim_device["Device_Type"]
    .map(device_control_map)
)

dim_device["Is_BYOD"] = (
    dim_device["Device_Type"]
    .eq("BYOD")
)

dim_device = dim_device[
    [
        "Device_Key",
        "Device_Type",
        "Device_Control_Type",
        "Is_BYOD"
    ]
]

display(dim_device)


,Device_Key,Device_Type,Device_Control_Type,Is_BYOD
0,1,BYOD,User Owned,True
1,2,Managed,Organization Managed,False


# 12. Dimension — Data Sensitivity


In [11]:
sensitivity_labels = {
    1: "Low",
    2: "Internal",
    3: "Confidential",
    4: "Personal / Restricted",
    5: "Critical / Highly Restricted"
}

dim_sensitivity = pd.DataFrame({
    "Data_Sensitivity": sorted(
        df["Data_Sensitivity"].dropna().unique()
    )
})

dim_sensitivity["Sensitivity_Key"] = np.arange(
    1,
    len(dim_sensitivity) + 1
)

dim_sensitivity["Sensitivity_Label"] = (
    dim_sensitivity["Data_Sensitivity"]
    .map(sensitivity_labels)
)

dim_sensitivity["High_Sensitivity_Flag"] = (
    dim_sensitivity["Data_Sensitivity"] >= 4
)

dim_sensitivity = dim_sensitivity[
    [
        "Sensitivity_Key",
        "Data_Sensitivity",
        "Sensitivity_Label",
        "High_Sensitivity_Flag"
    ]
]

display(dim_sensitivity)


,Sensitivity_Key,Data_Sensitivity,Sensitivity_Label,High_Sensitivity_Flag
0,1,1,Low,False
1,2,2,Internal,False
2,3,3,Confidential,False
3,4,4,Personal / Restricted,True
4,5,5,Critical / Highly Restricted,True


# 13. Dimension — Governance Rule


In [12]:
dim_rule = pd.DataFrame([
    ["AUTH-001", "Authorization", "No explicit permission", "BLOCK", 1],
    ["AUTH-002", "Authorization", "Role-action baseline = BLOCK", "BLOCK", 2],
    ["AUTH-003", "Authorization", "Role-action baseline = REVIEW", "REVIEW", 3],
    ["CTX-001", "Contextual Risk", "Critical contextual risk", "BLOCK", 4],
    ["CTX-002", "Contextual Risk", "High sensitivity + BYOD + high anomaly", "BLOCK", 5],
    ["CTX-003", "Contextual Risk", "High contextual risk", "REVIEW", 6],
    ["CTX-004", "Contextual Risk", "High sensitivity + failed login", "REVIEW", 7],
    ["CTX-005", "Contextual Risk", "Medium contextual risk", "REVIEW", 8],
    ["DEFAULT-001", "Default", "No higher-priority rule triggered", "ALLOW", 99]
], columns=[
    "Rule_ID",
    "Rule_Category",
    "Rule_Description",
    "Rule_Decision",
    "Rule_Priority"
])

dim_rule["Rule_Key"] = np.arange(
    1,
    len(dim_rule) + 1
)

dim_rule = dim_rule[
    [
        "Rule_Key",
        "Rule_ID",
        "Rule_Category",
        "Rule_Description",
        "Rule_Decision",
        "Rule_Priority"
    ]
]

display(dim_rule)


,Rule_Key,Rule_ID,Rule_Category,Rule_Description,Rule_Decision,Rule_Priority
0,1,AUTH-001,Authorization,No explicit permission,BLOCK,1
1,2,AUTH-002,Authorization,Role-action baseline = BLOCK,BLOCK,2
2,3,AUTH-003,Authorization,Role-action baseline = REVIEW,REVIEW,3
3,4,CTX-001,Contextual Risk,Critical contextual risk,BLOCK,4
4,5,CTX-002,Contextual Risk,High sensitivity + BYOD + high anomaly,BLOCK,5
5,6,CTX-003,Contextual Risk,High contextual risk,REVIEW,6
6,7,CTX-004,Contextual Risk,High sensitivity + failed login,REVIEW,7
7,8,CTX-005,Contextual Risk,Medium contextual risk,REVIEW,8
8,9,DEFAULT-001,Default,No higher-priority rule triggered,ALLOW,99


# 14. Dimension — Risk


In [13]:
risk_order = {
    "LOW": 1,
    "MEDIUM": 2,
    "HIGH": 3,
    "CRITICAL": 4
}

dim_risk = (
    df[["Contextual_Risk_Level"]]
    .drop_duplicates()
    .reset_index(drop=True)
)

dim_risk["Risk_Key"] = np.arange(
    1,
    len(dim_risk) + 1
)

dim_risk["Risk_Order"] = (
    dim_risk["Contextual_Risk_Level"]
    .map(risk_order)
)

dim_risk = (
    dim_risk[
        ["Risk_Key", "Contextual_Risk_Level", "Risk_Order"]
    ]
    .sort_values("Risk_Order")
    .reset_index(drop=True)
)

display(dim_risk)


,Risk_Key,Contextual_Risk_Level,Risk_Order
0,3,LOW,1
1,1,MEDIUM,2
2,2,HIGH,3
3,4,CRITICAL,4


# 15. Dimension — Access Hour

The source dataset has no full event date.

Therefore, the current time dimension is limited to hour-of-day analysis.


In [14]:
dim_hour = pd.DataFrame({
    "Access_Hour": sorted(
        df["Access_Hour"].dropna().unique()
    )
})

dim_hour["Hour_Key"] = np.arange(
    1,
    len(dim_hour) + 1
)

def hour_band(hour):
    if 0 <= hour < 6:
        return "Night"
    elif 6 <= hour < 12:
        return "Morning"
    elif 12 <= hour < 18:
        return "Afternoon"
    return "Evening"

dim_hour["Daypart"] = (
    dim_hour["Access_Hour"]
    .apply(hour_band)
)

dim_hour["Edge_Hour_Flag"] = (
    dim_hour["Access_Hour"].le(min_hour + 1) |
    dim_hour["Access_Hour"].ge(max_hour - 1)
)

dim_hour = dim_hour[
    [
        "Hour_Key",
        "Access_Hour",
        "Daypart",
        "Edge_Hour_Flag"
    ]
]

display(dim_hour)


,Hour_Key,Access_Hour,Daypart,Edge_Hour_Flag
0,1,6,Morning,True
1,2,7,Morning,True
2,3,8,Morning,False
3,4,9,Morning,False
4,5,10,Morning,False
5,6,11,Morning,False
6,7,12,Afternoon,False
7,8,13,Afternoon,False
8,9,14,Afternoon,False
9,10,15,Afternoon,False


# 16. Dimension — Region


In [15]:
dim_region = (
    df[["Region"]]
    .drop_duplicates()
    .sort_values("Region")
    .reset_index(drop=True)
)

dim_region["Region_Key"] = np.arange(
    1,
    len(dim_region) + 1
)

dim_region = dim_region[
    ["Region_Key", "Region"]
]

display(dim_region)


,Region_Key,Region
0,1,East
1,2,North
2,3,South
3,4,West


# 17. Dimension — Lead Source


In [16]:
dim_lead_source = (
    df[["Lead_Source"]]
    .drop_duplicates()
    .sort_values("Lead_Source")
    .reset_index(drop=True)
)

dim_lead_source["Lead_Source_Key"] = np.arange(
    1,
    len(dim_lead_source) + 1
)

dim_lead_source = dim_lead_source[
    ["Lead_Source_Key", "Lead_Source"]
]

display(dim_lead_source)


,Lead_Source_Key,Lead_Source
0,1,Campaign
1,2,Email
2,3,Partner
3,4,Referral
4,5,Web


# 18. Create Synthetic Privacy-Safe Customer Dimension

Stage 6 introduced a synthetic customer layer.

For the analytical model, we create a small privacy-safe customer dimension that can support future CRM/customer governance analysis.


In [17]:
n_customers = 10000

customer_ids = [
    f"CUST_{i:06d}"
    for i in range(1, n_customers + 1)
]

states = [
    "SP", "RJ", "MG", "DF", "PR",
    "RS", "BA", "SC", "GO", "PE"
]

segments = [
    "New Customer",
    "Regular",
    "High Value",
    "At Risk",
    "Inactive"
]

signup_dates = pd.to_datetime(
    np.random.choice(
        pd.date_range("2022-01-01", "2026-08-01", freq="D"),
        size=n_customers
    )
)

birth_dates = pd.to_datetime(
    np.random.choice(
        pd.date_range("1955-01-01", "2005-12-31", freq="D"),
        size=n_customers
    )
)

def pseudonymize_identifier(value, salt="crm-governance-demo"):
    raw_value = f"{salt}|{value}"
    return hashlib.sha256(
        raw_value.encode("utf-8")
    ).hexdigest()[:16]

customer_base = pd.DataFrame({
    "Customer_ID": customer_ids,
    "State": np.random.choice(states, n_customers),
    "Signup_Date": signup_dates,
    "Birth_Date": birth_dates,
    "Marketing_Consent": np.random.choice(
        [True, False],
        n_customers,
        p=[0.72, 0.28]
    ),
    "Customer_Segment": np.random.choice(
        segments,
        n_customers,
        p=[0.15, 0.45, 0.15, 0.15, 0.10]
    ),
    "Purchase_Count": np.random.poisson(8, n_customers),
    "Total_Revenue": np.round(
        np.random.gamma(2.5, 450, n_customers),
        2
    )
})

customer_base["Customer_Token"] = (
    customer_base["Customer_ID"]
    .apply(pseudonymize_identifier)
)

REFERENCE_DATE = pd.Timestamp("2026-08-28")

customer_base["Age"] = (
    (
        REFERENCE_DATE -
        customer_base["Birth_Date"]
    ).dt.days / 365.25
).astype(int)

customer_base["Age_Group"] = pd.cut(
    customer_base["Age"],
    bins=[0, 24, 34, 44, 54, 64, 200],
    labels=[
        "18-24",
        "25-34",
        "35-44",
        "45-54",
        "55-64",
        "65+"
    ],
    include_lowest=True
)

customer_base["Average_Ticket"] = np.where(
    customer_base["Purchase_Count"] > 0,
    customer_base["Total_Revenue"] /
    customer_base["Purchase_Count"],
    0
).round(2)

dim_customer = customer_base[
    [
        "Customer_Token",
        "Age_Group",
        "State",
        "Signup_Date",
        "Marketing_Consent",
        "Customer_Segment",
        "Purchase_Count",
        "Total_Revenue",
        "Average_Ticket"
    ]
].copy()

dim_customer["Customer_Key"] = np.arange(
    1,
    len(dim_customer) + 1
)

dim_customer = dim_customer[
    [
        "Customer_Key",
        "Customer_Token",
        "Age_Group",
        "State",
        "Signup_Date",
        "Marketing_Consent",
        "Customer_Segment",
        "Purchase_Count",
        "Total_Revenue",
        "Average_Ticket"
    ]
]

display(dim_customer.head())


,Customer_Key,Customer_Token,Age_Group,State,Signup_Date,Marketing_Consent,Customer_Segment,Purchase_Count,Total_Revenue,Average_Ticket
0,1,7b91ee80bfbdf1d4,35-44,RJ,2025-01-31,True,At Risk,13,1415.27,108.87
1,2,f6b2732f9c4741b8,25-34,BA,2025-12-30,True,At Risk,9,2225.68,247.30
2,3,a5018f78abdc7e1e,18-24,RJ,2024-05-10,True,New Customer,6,1507.28,251.21
3,4,0665cc2f3715bf80,35-44,DF,2025-07-18,False,High Value,9,724.80,80.53
4,5,223e2270ecaedfcd,55-64,SP,2025-02-04,True,At Risk,5,1994.14,398.83


# 19. Build the Access Fact Table

The fact table contains event-level metrics plus foreign keys to dimensions.


In [18]:
fact_access = df.copy()

fact_access = (
    fact_access
    .merge(dim_role, on="Role", how="left")
    .merge(dim_action, on="CRM_Action", how="left")
    .merge(dim_device, on="Device_Type", how="left")
    .merge(
        dim_sensitivity,
        on="Data_Sensitivity",
        how="left",
        suffixes=("", "_dim")
    )
    .merge(
        dim_rule[
            ["Rule_Key", "Rule_ID"]
        ],
        left_on="Triggered_Rule_ID",
        right_on="Rule_ID",
        how="left"
    )
    .merge(
        dim_risk[
            ["Risk_Key", "Contextual_Risk_Level"]
        ],
        on="Contextual_Risk_Level",
        how="left"
    )
    .merge(dim_hour, on="Access_Hour", how="left", suffixes=("", "_dim"))
    .merge(dim_region, on="Region", how="left")
    .merge(dim_lead_source, on="Lead_Source", how="left")
)

fact_access["Access_Count"] = 1
fact_access["Proposed_Block_Flag"] = (
    fact_access["Proposed_Access_Decision"]
    .eq("BLOCK")
    .astype(int)
)
fact_access["Review_Flag"] = (
    fact_access["Proposed_Access_Decision"]
    .eq("REVIEW")
    .astype(int)
)
fact_access["Allow_Flag"] = (
    fact_access["Proposed_Access_Decision"]
    .eq("ALLOW")
    .astype(int)
)

fact_access = fact_access[
    [
        "Access_Event_Key",
        "Role_Key",
        "Action_Key",
        "Device_Key",
        "Sensitivity_Key",
        "Rule_Key",
        "Risk_Key",
        "Hour_Key",
        "Region_Key",
        "Lead_Source_Key",
        "User_ID",
        "Daily_Logins",
        "Failed_Logins",
        "Policy_Compliance_Score",
        "Anomaly_Score",
        "Governance_Score",
        "Permission_Granted",
        "Baseline_Authorization",
        "Contextual_Risk_Score",
        "Access_Decision",
        "Proposed_Access_Decision",
        "Failed_Rule_Count",
        "DQ_Status",
        "Access_Count",
        "Is_Blocked",
        "Proposed_Block_Flag",
        "Review_Flag",
        "Allow_Flag"
    ]
]

display(fact_access.head())


,Access_Event_Key,Role_Key,Action_Key,Device_Key,Sensitivity_Key,Rule_Key,Risk_Key,Hour_Key,Region_Key,Lead_Source_Key,User_ID,Daily_Logins,Failed_Logins,Policy_Compliance_Score,Anomaly_Score,Governance_Score,Permission_Granted,Baseline_Authorization,Contextual_Risk_Score,Access_Decision,Proposed_Access_Decision,Failed_Rule_Count,DQ_Status,Access_Count,Is_Blocked,Proposed_Block_Flag,Review_Flag,Allow_Flag
0,1,1,6,2,5,7,1,5,2,4,1,6,1,76.19,0.114,51.81,True,ALLOW,2,Block,REVIEW,0,PASS,1,1,0,1,0
1,2,4,6,2,5,6,2,7,4,5,2,5,1,62.87,0.464,38.40,True,ALLOW,5,Block,REVIEW,0,PASS,1,1,0,1,0
2,3,4,4,2,1,1,3,15,3,3,3,6,1,81.87,0.083,48.57,False,ALLOW,1,Block,BLOCK,0,PASS,1,1,1,0,0
3,4,2,6,2,3,9,3,5,3,1,4,10,0,85.89,0.181,57.53,True,ALLOW,0,Block,ALLOW,0,PASS,1,1,0,0,1
4,5,4,4,2,1,1,1,12,4,5,5,6,2,92.31,0.309,44.58,False,ALLOW,2,Block,BLOCK,0,PASS,1,1,1,0,0


# 20. Star Schema Overview

```text
                    DIM_ROLE
                       |
                    DIM_ACTION
                       |
                   DIM_DEVICE
                       |
               DIM_SENSITIVITY
                       |
                    DIM_RULE
                       |
                    DIM_RISK
                       |
                    DIM_HOUR
                       |
                   DIM_REGION
                       |
                DIM_LEAD_SOURCE
                       |
                       v
                FACT_ACCESS_EVENT


            DIM_CUSTOMER_PRIVACY_SAFE
                       |
                       v
          CUSTOMER GOVERNANCE ANALYTICS
```

The customer dimension remains logically separate because the current access dataset has no real customer key linking access events to customer records.


# 21. Referential Integrity Checks

Each fact foreign key should successfully resolve to a dimension row.


In [19]:
foreign_keys = [
    "Role_Key",
    "Action_Key",
    "Device_Key",
    "Sensitivity_Key",
    "Rule_Key",
    "Risk_Key",
    "Hour_Key",
    "Region_Key",
    "Lead_Source_Key"
]

referential_integrity = pd.DataFrame({
    "Foreign_Key": foreign_keys,
    "Missing_Key_Count": [
        fact_access[key].isna().sum()
        for key in foreign_keys
    ]
})

display(referential_integrity)


,Foreign_Key,Missing_Key_Count
0,Role_Key,0
1,Action_Key,0
2,Device_Key,0
3,Sensitivity_Key,0
4,Rule_Key,0
5,Risk_Key,0
6,Hour_Key,0
7,Region_Key,0
8,Lead_Source_Key,0


# 22. Fact Table Validation

The fact table should preserve the original event count.


In [20]:
validation_summary = pd.DataFrame({
    "Metric": [
        "Original Access Rows",
        "Fact Access Rows",
        "Unique Event Keys",
        "Duplicate Event Keys"
    ],
    "Value": [
        len(access),
        len(fact_access),
        fact_access["Access_Event_Key"].nunique(),
        fact_access["Access_Event_Key"].duplicated().sum()
    ]
})

display(validation_summary)


,Metric,Value
0,Original Access Rows,50000
1,Fact Access Rows,50000
2,Unique Event Keys,50000
3,Duplicate Event Keys,0


# 23. Analytical Measures — Prototype

These measures are calculated in pandas for validation.

Equivalent DAX measures can later be created in Power BI.


In [21]:
total_access = fact_access["Access_Count"].sum()

blocked_access_rate = (
    fact_access["Is_Blocked"].mean() * 100
)

proposed_block_rate = (
    fact_access["Proposed_Block_Flag"].mean() * 100
)

review_rate = (
    fact_access["Review_Flag"].mean() * 100
)

allow_rate = (
    fact_access["Allow_Flag"].mean() * 100
)

critical_risk_rate = (
    fact_access["Risk_Key"]
    .isin(
        dim_risk.loc[
            dim_risk["Contextual_Risk_Level"] == "CRITICAL",
            "Risk_Key"
        ]
    )
    .mean() * 100
)

dq_pass_rate = (
    fact_access["DQ_Status"]
    .eq("PASS")
    .mean() * 100
)

measure_validation = pd.DataFrame({
    "Measure": [
        "Total Access Events",
        "Original Block Rate %",
        "Proposed Block Rate %",
        "Review Rate %",
        "Allow Rate %",
        "Critical Risk Event Rate %",
        "DQ Pass Rate %"
    ],
    "Value": [
        total_access,
        blocked_access_rate,
        proposed_block_rate,
        review_rate,
        allow_rate,
        critical_risk_rate,
        dq_pass_rate
    ]
})

display(measure_validation.round(2))


,Measure,Value
0,Total Access Events,50000.00
1,Original Block Rate %,84.14
2,Proposed Block Rate %,38.78
3,Review Rate %,31.20
4,Allow Rate %,30.02
5,Critical Risk Event Rate %,0.46
6,DQ Pass Rate %,100.00


# 24. Governance Metrics by Role


In [22]:
role_metrics = (
    fact_access
    .merge(dim_role, on="Role_Key", how="left")
    .groupby("Role")
    .agg(
        Access_Events=("Access_Count", "sum"),
        Proposed_Block_Rate=("Proposed_Block_Flag", "mean"),
        Review_Rate=("Review_Flag", "mean"),
        Avg_Risk_Score=("Contextual_Risk_Score", "mean"),
        Avg_Anomaly_Score=("Anomaly_Score", "mean"),
        DQ_Pass_Rate=("DQ_Status", lambda x: x.eq("PASS").mean())
    )
    .reset_index()
)

for col in [
    "Proposed_Block_Rate",
    "Review_Rate",
    "DQ_Pass_Rate"
]:
    role_metrics[col] *= 100

display(role_metrics.round(2))


,Role,Access_Events,Proposed_Block_Rate,Review_Rate,Avg_Risk_Score,Avg_Anomaly_Score,DQ_Pass_Rate
0,Admin,4982,1.81,49.04,1.67,0.22,100.00
1,Analyst,7389,64.91,17.61,1.92,0.22,100.00
2,Manager,9904,3.48,51.81,1.67,0.22,100.00
3,Sales Rep,20159,45.84,26.99,1.90,0.22,100.00
4,Support,7566,65.03,16.98,1.91,0.22,99.99


# 25. Governance Metrics by CRM Action


In [23]:
action_metrics = (
    fact_access
    .merge(
        dim_action[
            ["Action_Key", "CRM_Action", "Action_Risk_Level"]
        ],
        on="Action_Key",
        how="left"
    )
    .groupby(
        ["CRM_Action", "Action_Risk_Level"]
    )
    .agg(
        Access_Events=("Access_Count", "sum"),
        Proposed_Block_Rate=("Proposed_Block_Flag", "mean"),
        Review_Rate=("Review_Flag", "mean"),
        Avg_Risk_Score=("Contextual_Risk_Score", "mean"),
        Avg_Anomaly_Score=("Anomaly_Score", "mean")
    )
    .reset_index()
)

action_metrics["Proposed_Block_Rate"] *= 100
action_metrics["Review_Rate"] *= 100

display(
    action_metrics
    .sort_values("Proposed_Block_Rate", ascending=False)
    .round(2)
)


,CRM_Action,Action_Risk_Level,Access_Events,Proposed_Block_Rate,Review_Rate,Avg_Risk_Score,Avg_Anomaly_Score
2,DeleteLead,Critical,987,90.88,3.95,2.15,0.22
0,ApproveDiscount,High,5081,72.45,14.27,2.06,0.22
3,EditLead,Medium,12293,70.40,14.81,2.07,0.22
4,ExportCRM,High,4019,70.04,25.43,2.05,0.21
1,CreateOpportunity,Medium,9924,30.98,33.97,1.66,0.22
5,ViewLead,Low,17696,1.53,48.72,1.64,0.22


# 26. Privacy-Safe Customer Metrics

These measures validate the customer dimension for future privacy and CRM analytics.


In [24]:
customer_metrics = pd.DataFrame({
    "Metric": [
        "Customers",
        "Marketing Consent Rate %",
        "Average Purchase Count",
        "Average Revenue",
        "Average Ticket"
    ],
    "Value": [
        len(dim_customer),
        dim_customer["Marketing_Consent"].mean() * 100,
        dim_customer["Purchase_Count"].mean(),
        dim_customer["Total_Revenue"].mean(),
        dim_customer["Average_Ticket"].mean()
    ]
})

display(customer_metrics.round(2))


,Metric,Value
0,Customers,10000.00
1,Marketing Consent Rate %,73.05
2,Average Purchase Count,7.98
3,Average Revenue,1132.97
4,Average Ticket,164.79


# 27. Power BI Table Inventory

The following tables are intended for the Power BI semantic model.


In [25]:
powerbi_table_inventory = pd.DataFrame([
    ["fact_access_event", "Fact", len(fact_access), "CRM access-event grain"],
    ["dim_role", "Dimension", len(dim_role), "User role"],
    ["dim_action", "Dimension", len(dim_action), "CRM action"],
    ["dim_device", "Dimension", len(dim_device), "Device context"],
    ["dim_sensitivity", "Dimension", len(dim_sensitivity), "Data sensitivity"],
    ["dim_rule", "Dimension", len(dim_rule), "Governance rule"],
    ["dim_risk", "Dimension", len(dim_risk), "Contextual risk"],
    ["dim_hour", "Dimension", len(dim_hour), "Access hour and daypart"],
    ["dim_region", "Dimension", len(dim_region), "Region"],
    ["dim_lead_source", "Dimension", len(dim_lead_source), "Lead source"],
    ["dim_customer", "Dimension", len(dim_customer), "Privacy-safe customer analytics"]
], columns=[
    "Table_Name",
    "Table_Type",
    "Rows",
    "Purpose"
])

display(powerbi_table_inventory)


,Table_Name,Table_Type,Rows,Purpose
0,fact_access_event,Fact,50000,CRM access-event grain
1,dim_role,Dimension,5,User role
2,dim_action,Dimension,6,CRM action
3,dim_device,Dimension,2,Device context
4,dim_sensitivity,Dimension,5,Data sensitivity
5,dim_rule,Dimension,9,Governance rule
6,dim_risk,Dimension,4,Contextual risk
7,dim_hour,Dimension,17,Access hour and daypart
8,dim_region,Dimension,4,Region
9,dim_lead_source,Dimension,5,Lead source


# 28. Suggested Power BI Relationships

Recommended relationship pattern:

| From | To | Cardinality | Direction |
|---|---|---|---|
| dim_role[Role_Key] | fact_access_event[Role_Key] | 1:* | Single |
| dim_action[Action_Key] | fact_access_event[Action_Key] | 1:* | Single |
| dim_device[Device_Key] | fact_access_event[Device_Key] | 1:* | Single |
| dim_sensitivity[Sensitivity_Key] | fact_access_event[Sensitivity_Key] | 1:* | Single |
| dim_rule[Rule_Key] | fact_access_event[Rule_Key] | 1:* | Single |
| dim_risk[Risk_Key] | fact_access_event[Risk_Key] | 1:* | Single |
| dim_hour[Hour_Key] | fact_access_event[Hour_Key] | 1:* | Single |
| dim_region[Region_Key] | fact_access_event[Region_Key] | 1:* | Single |
| dim_lead_source[Lead_Source_Key] | fact_access_event[Lead_Source_Key] | 1:* | Single |

The privacy-safe customer dimension remains separate until a governed business key exists to link access events to customer records.


# 29. Suggested DAX Measures

Examples for the future Power BI model:

```DAX
Total Access Events =
SUM ( fact_access_event[Access_Count] )
```

```DAX
Proposed Block Rate =
DIVIDE (
    SUM ( fact_access_event[Proposed_Block_Flag] ),
    [Total Access Events]
)
```

```DAX
Review Rate =
DIVIDE (
    SUM ( fact_access_event[Review_Flag] ),
    [Total Access Events]
)
```

```DAX
DQ Pass Rate =
DIVIDE (
    CALCULATE (
        COUNTROWS ( fact_access_event ),
        fact_access_event[DQ_Status] = "PASS"
    ),
    [Total Access Events]
)
```

```DAX
Critical Risk Events =
CALCULATE (
    [Total Access Events],
    dim_risk[Contextual_Risk_Level] = "CRITICAL"
)
```


# 30. Exportable Analytical Model Artifacts

The following DataFrames are intended to become persisted CSV or parquet outputs:

- `fact_access`
- `dim_role`
- `dim_action`
- `dim_device`
- `dim_sensitivity`
- `dim_rule`
- `dim_risk`
- `dim_hour`
- `dim_region`
- `dim_lead_source`
- `dim_customer`
- `powerbi_table_inventory`

Suggested GitHub structure:

```text
models/
├── fact_access_event.csv
├── dim_role.csv
├── dim_action.csv
├── dim_device.csv
├── dim_sensitivity.csv
├── dim_rule.csv
├── dim_risk.csv
├── dim_hour.csv
├── dim_region.csv
├── dim_lead_source.csv
└── dim_customer_privacy_safe.csv
```


# 31. Findings to Document

## Fact Table

- **Grain:** One row per CRM access event.

- **Row count:** 50,000 access events.

- **Foreign-key completeness:** 100%. All foreign keys in the fact table successfully resolve to the corresponding dimensions, with zero missing keys across Role, Action, Device, Sensitivity, Rule, Risk, Hour, Region, and Lead Source.

- **Duplicate event keys:** 0. All 50,000 `Access_Event_Key` values are unique.

## Dimensions

- **Number of dimensions:** 10 dimensions are included in the analytical model.

- **Highest-cardinality dimension:** `dim_customer`, with 10,000 rows. Among the dimensions directly related to the access fact table, `dim_hour` has the highest cardinality, with 17 rows.

- **Governance-specific dimensions:** `dim_sensitivity`, `dim_rule`, and `dim_risk` are the main governance-specific dimensions. `dim_device` also supports governance analysis by representing managed versus BYOD access context.

## Governance Metrics

- **Proposed Block Rate:** 38.78%.

- **Review Rate:** 31.20%.

- **Critical Risk Rate:** 0.46%.

- **DQ Pass Rate:** 100.00% when rounded to two decimal places.

## Privacy-Safe Customer Layer

- **Customers:** 10,000.

- **Consent Rate:** 73.05%.

- **Direct identifiers exposed:** 0 direct customer identifiers are exposed in the privacy-safe analytical dimension. `Customer_ID` is replaced by `Customer_Token`, while direct identifying attributes are excluded.

## Modeling Conclusions

1. **The analytical model successfully converts the governed CRM dataset into a star-schema structure with a clearly defined event-level fact table, complete referential integrity, and reusable dimensions for governance analysis.**

2. **Governance concepts are embedded directly into the semantic model through dedicated dimensions for rules, contextual risk, and data sensitivity, allowing access decisions, Data Quality, and risk indicators to be analyzed consistently across business dimensions.**

3. **The privacy-safe customer dimension remains intentionally separate from the access-event fact table because no governed business key currently exists between the two datasets; this avoids creating an artificial relationship and preserves both analytical integrity and privacy-by-design principles.**

# 32. Limitations

1. The source access dataset contains no full event date.
2. There is no real customer key linking access events to customer records.
3. The customer dimension is synthetic.
4. Governance rules are project-defined.
5. The star schema is designed for analytical monitoring, not operational transaction processing.
6. Slowly Changing Dimensions are not implemented.
7. Historical rule versioning is not implemented yet.
8. Power BI relationships are documented but not physically created in this notebook.


# 33. Next Step — Governance Monitoring in Power BI

## Stage 10 — Governance Monitoring / Power BI

Planned outputs:

- Governance Overview page;
- Access Governance page;
- Data Quality page;
- Privacy Monitoring page;
- Metadata & Lineage page;
- Governance Operations page;
- KPI definitions;
- DAX measures;
- drill-through and filtering strategy;
- dashboard storytelling and executive summary.

This stage will convert the governance framework into a visual monitoring product.
